# MC Dropout — Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning (2016)

_Papers · Meta-learning, Bayesian & Robustness_

**Leave dropout ON at test time, run the network many times, and the spread of the answers measures how unsure the model is.**

---

This notebook is a practice scaffold for the **MC Dropout — Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn
import numpy as np
torch.manual_seed(1); np.random.seed(1)

# --- 0. Sanity-check the worked example: MC dropout mean/variance over 4 passes by hand. ---
# 4 hidden activations h=[1,2,3,4], output weights all 1, no bias. Dropout p=0.5 -> inverted
# scale 1/(1-p)=2. Four masks keep [1,2], [2,3], [1,4], [3,4]; output = (sum of kept) * 2.
h = np.array([1., 2., 3., 4.]); scale = 2.0
masks = [np.array([1,1,0,0]), np.array([0,1,1,0]),
         np.array([1,0,0,1]), np.array([0,0,1,1])]
outs = np.array([ (h * m * scale).sum() for m in masks ])
print("worked example: outs =", list(outs))
print("  predictive mean =", outs.mean(),
      " variance =", outs.var(), " std = %.4f" % outs.std())
# worked example: outs = [6.0, 10.0, 10.0, 14.0]
#   predictive mean = 10.0  variance = 8.0  std = 2.8284


# --- 1. The model: a small multi-layer perceptron with dropout, composed with torch.nn. ---
P = 0.2
def make_net():
    return nn.Sequential(nn.Linear(1, 100), nn.ReLU(), nn.Dropout(P),
                         nn.Linear(100, 100), nn.ReLU(), nn.Dropout(P),
                         nn.Linear(100, 1))

# --- 2. A 1-D regression problem with NO data outside [-4, 4] (so |x|>4 is extrapolation). ---
def f(x): return np.sin(1.5 * x)
xa = np.random.uniform(-4.0, -1.5, 50)
xb = np.random.uniform( 1.5,  4.0, 50)
xtr = np.concatenate([xa, xb]).astype(np.float32)
ytr = (f(xtr) + 0.03 * np.random.randn(len(xtr))).astype(np.float32)
Xtr = torch.tensor(xtr).unsqueeze(1); Ytr = torch.tensor(ytr).unsqueeze(1)

# --- 3. Train normally (dropout ON during training, as usual). ---
net = make_net()
opt = torch.optim.Adam(net.parameters(), lr=5e-3, weight_decay=1e-5)
lossf = nn.MSELoss()
net.train()
for epoch in range(1500):
    opt.zero_grad(); loss = lossf(net(Xtr), Ytr); loss.backward(); opt.step()
print("\nfinal train loss: %.5f" % loss.item())

# --- 4. MC DROPOUT (the novel part): keep dropout ON, T stochastic forward passes. ---
def mc_dropout(net, x, T=200):
    net.train()                                  # <-- dropout STAYS ON (do NOT call net.eval())
    with torch.no_grad():
        preds = torch.stack([net(x) for _ in range(T)], dim=0)   # (T, N, 1)
    return preds.mean(0), preds.std(0)           # predictive mean (Eq. 6), uncertainty

# --- 5. Compare uncertainty INSIDE the data range vs FAR outside it. ---
for xq in [-3.0, 3.0, 7.0, -7.0]:
    mean, std = mc_dropout(net, torch.tensor([[xq]]))
    tag = "in-data    " if abs(xq) <= 4 else "extrapolate"
    print("  x=%5.1f (%s): pred=%6.3f  uncertainty(std)=%.4f" % (
          xq, tag, mean.item(), std.item()))

# final train loss: ~0.0216
#   x= -3.0 (in-data    ): pred=~0.89   uncertainty(std)=~0.112   <- low: data is dense here
#   x=  3.0 (in-data    ): pred=~-0.93  uncertainty(std)=~0.107   <- low
#   x=  7.0 (extrapolate): pred=~-0.24  uncertainty(std)=~0.403   <- HIGH: far from any data
#   x= -7.0 (extrapolate): pred=~-0.32  uncertainty(std)=~0.413   <- HIGH
# (Our small run, not the paper's reported numbers. Exact values vary by seed/hardware.)

# --- 6. Ablation: net.eval() turns dropout OFF -> all T passes identical -> uncertainty = 0. ---
net.eval()
with torch.no_grad():
    e = torch.stack([net(torch.tensor([[7.0]])) for _ in range(200)], 0)
print("\nablation net.eval() at x=7.0: std across 200 passes = %.6f (signal gone)" % e.std().item())
# ablation net.eval() at x=7.0: std across 200 passes = 0.000000 (signal gone)

## Visualize the data & results

_As the test input moves from inside the training data range out to where no data was seen, how does MC dropout's predictive uncertainty (std across T=200 passes) change?_

In [ ]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(1); np.random.seed(1)

# 1-D regression, data only in two bands; nothing outside [-4, 4].
def f(x): return np.sin(1.5 * x)
xtr = np.concatenate([np.random.uniform(-4.0,-1.5,50),
                      np.random.uniform( 1.5, 4.0,50)]).astype(np.float32)
ytr = (f(xtr) + 0.03*np.random.randn(len(xtr))).astype(np.float32)
Xtr = torch.tensor(xtr).unsqueeze(1); Ytr = torch.tensor(ytr).unsqueeze(1)

P = 0.2
net = nn.Sequential(nn.Linear(1,100), nn.ReLU(), nn.Dropout(P),
                    nn.Linear(100,100), nn.ReLU(), nn.Dropout(P),
                    nn.Linear(100,1))
opt = torch.optim.Adam(net.parameters(), lr=5e-3, weight_decay=1e-5); lf = nn.MSELoss()
net.train()
for e in range(1500):
    opt.zero_grad(); lf(net(Xtr), Ytr).backward(); opt.step()

# MC dropout: keep dropout ON, T=200 passes across a grid of inputs.
net.train(); T = 200
grid = torch.linspace(-8, 8, 161).unsqueeze(1)
with torch.no_grad():
    preds = torch.stack([net(grid) for _ in range(T)], 0)   # (T, 161, 1)
mean = preds.mean(0).squeeze(1).numpy()
std  = preds.std(0).squeeze(1).numpy()
xs = np.linspace(-8, 8, 161)

indata = std[((xs>=-4)&(xs<=-1.5)) | ((xs>=1.5)&(xs<=4))]
extrap = std[(xs<=-6) | (xs>=6)]
print("mean std IN-DATA   : %.3f" % indata.mean())
print("mean std EXTRAPOL  : %.3f" % extrap.mean())
sel = list(range(0, 161, 8))
print([[round(float(xs[i]),1), round(float(std[i]),3)] for i in sel])
# mean std IN-DATA   : 0.105
# mean std EXTRAPOL  : 0.397
# std stays ~0.09-0.14 inside the data range and climbs to ~0.4-0.5 past |x|=7.
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The eval() ablation. You have a working MC-dropout setup that keeps the network in
            net.train() for the $T$ passes. Switch it to net.eval() (everything else
            identical) and re-measure the uncertainty. What happens to the spread across the $T$ passes, and
            why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what net.eval() does to nn.Dropout: it turns dropout OFF (no units are zeroed; activations pass through unscaled). — _In eval mode dropout is the identity, so the network becomes deterministic for a fixed input._
- Run the $T$ forward passes in eval mode and look at the outputs: all $T$ outputs are byte-for-byte identical. — _With no random mask, the same input always yields the same output, so there is nothing to vary across passes._
- Compute the variance / standard deviation of $T$ identical numbers. — _The variance of a list of identical values is exactly $0$ &mdash; the uncertainty signal vanishes._

**Answer:** In net.eval() mode dropout is OFF, so all $T$ forward passes return the
                 same number. Their variance is exactly $0$ and the standard deviation is $0$ everywhere
                 &mdash; including far outside the training data, where you most wanted a warning. The whole
                 uncertainty estimate collapses. This is exactly why MC dropout's defining instruction is "keep
                 dropout ON at test time": the random masks are the source of the spread you measure.

</details>

**Problem 2.** Why is the MC-dropout uncertainty low near the training data and high far outside the
            data range, even though it is the same network with the same dropout rate everywhere?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Think about a test point sitting on top of training points. During training, however the mask fell, the network was pushed to reproduce the seen target there. — _The output near data is constrained by the loss, so different masks must all give nearly the same answer &mdash; small spread._
- Now think about a test point far outside the training range. No training point ever constrained the output there. — _Nothing pins the output down, so different masks send it in different directions &mdash; large spread._
- Connect spread to uncertainty: spread across masks is the predictive variance, and masks are posterior samples (&sect;3). — _A large posterior sample spread is precisely the model saying "many weight settings consistent with my training disagree here" &mdash; epistemic uncertainty._

**Answer:** The dropout masks are samples from the model's approximate weight posterior. Where training
                 data pins the output down, every mask must reproduce it, so the $T$ runs agree &mdash; small
                 spread, low uncertainty. Where no data constrains the output (outside the training range),
                 different masks pull the answer in different directions, so the $T$ runs scatter &mdash; large
                 spread, high uncertainty. Same network, same rate; the difference is whether the data
                 constrained that region. In our run the mean standard deviation is about $0.10$ inside the data
                 range and about $0.40$ in the extrapolation tails &mdash; roughly four times larger.

</details>

**Problem 3.** In the worked example you had hidden activations $h = [1, 2, 3, 4]$, output weights all $1$, dropout
            $p = 0.5$ (inverted-dropout scale $2$), and the four passes kept $[1,2]$, $[2,3]$, $[1,4]$, $[3,4]$,
            giving outputs $[6, 10, 10, 14]$ with mean $10.0$ and variance $8.0$. Suppose a different input gives
            four passes with outputs $[9, 11, 9, 11]$ instead. Compute its mean and standard deviation, and say
            which input the model is more confident about.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean of $[9, 11, 9, 11]$: $\tfrac{1}{4}(9+11+9+11) = \tfrac{40}{4} = 10.0$. — _Same predictive mean as the first input ($10.0$), so the point estimate is identical._
- Deviations from the mean: $[-1, 1, -1, 1]$; squared $[1, 1, 1, 1]$; variance $= \tfrac{4}{4} = 1.0$; standard deviation $= \sqrt{1.0} = 1.0$. — _The four outputs are tightly clustered, so the spread is small._
- Compare spreads: first input std $\approx 2.83$, second input std $= 1.0$. — _Same mean, very different uncertainty &mdash; the spread, not the mean, carries the confidence._

**Answer:** The second input has mean $10.0$ and standard deviation $1.0$ &mdash; the same
                 prediction as the first input but a much smaller spread ($1.0$ versus $\approx 2.83$). The
                 model is therefore more confident about the second input. This is the heart of MC
                 dropout: two inputs can share a prediction yet differ in trustworthiness, and only the
                 across-pass spread reveals it.

</details>